# D1 Anomaly Baseline Results

Purpose: review the first real TyreVisionX tyre anomaly baseline.

Method: train on good-only D1 images, calibrate threshold on mixed validation, evaluate once on mixed test.

Current status: completed with cached ResNet18 ImageNet weights and Mahalanobis scoring. This is a baseline for professor discussion, not a production detector.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown

RUN_DIR = Path("artifacts/anomaly/d1_resnet18_mahalanobis_v1")
print("Run dir exists:", RUN_DIR.exists())
metadata = json.loads((RUN_DIR / "metadata.json").read_text())
metrics = json.loads((RUN_DIR / "metrics_test.json").read_text())
display(metadata["feature_extractor"])
display(metrics)

In [ ]:
tm = metrics["threshold_metrics"]
summary = pd.DataFrame([
    {"metric": "AUROC", "value": metrics["auroc"]},
    {"metric": "AUPRC", "value": metrics["auprc"]},
    {"metric": "anomaly recall", "value": tm["recall"]},
    {"metric": "anomaly precision", "value": tm["precision"]},
    {"metric": "normal FPR", "value": tm["normal_fpr"]},
    {"metric": "false negatives", "value": tm["fn"]},
    {"metric": "false positives", "value": tm["fp"]},
])
display(summary)

In [ ]:
for name in ["confusion_matrix_test.png", "score_distribution_test.png", "pr_curve_test.png"]:
    img = RUN_DIR / name
    if img.exists():
        display(Markdown(f"## {name}"))
        display(Image(filename=str(img)))
    else:
        print("Missing", img)

In [ ]:
pred = pd.read_csv(RUN_DIR / "predictions_test.csv")
fn = pd.read_csv(RUN_DIR / "false_negatives_test.csv")
fp = pd.read_csv(RUN_DIR / "false_positives_test.csv")
print("predictions", len(pred), "false negatives", len(fn), "false positives", len(fp))
display(fn.head(20))

## Interpretation

The baseline ran correctly and used validation-only thresholding. The main limitation is low defect/anomaly recall on test: many defective tyres scored below the selected threshold. The next research step is visual false-negative review and comparison against kNN or patch-level features.